<a href="https://colab.research.google.com/github/jmlee119/lotto_prediction/blob/main/%EB%A1%9C%EB%98%90%EB%B2%88%ED%98%B8%EC%98%88%EC%B8%A1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install tensorflow pandas numpy scikit-learn

# 로또 번호 생성기

In [4]:
import numpy as np
import pandas as pd
from google.colab import drive
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input

# ======================
# 0. Google Drive 연결
# ======================
drive.mount('/content/drive')

# ======================
# 1. 원본 데이터 로드 & 전처리
# ======================
# Drive에 저장된 lotto.csv 읽기
raw_df = pd.read_csv("/content/drive/MyDrive/lotto.csv", header=None)

# 1~6번 컬럼 (당첨번호만)
df = raw_df.iloc[:, 1:7]
df.columns = ['n1', 'n2', 'n3', 'n4', 'n5', 'n6']

print("전처리 데이터:")
print(df.head())

# ======================
# 2. One-hot 변환 (핵심)
# ======================
def to_onehot(row):
    arr = np.zeros(45)
    for num in row:
        arr[int(num)-1] = 1
    return arr

data = np.array([to_onehot(row) for row in df.values])

# ======================
# 3. 시퀀스 데이터 생성
# ======================
seq_length = 10

X = []
y = []

for i in range(len(data) - seq_length):
    X.append(data[i:i+seq_length])
    y.append(data[i+seq_length])

X = np.array(X)
y = np.array(y)

# ======================
# X : 학습 데이터 개수 / 입력 시퀀스의 수 / 번호 개수
# Y : 정답 확률 / 번호 개수
# ======================
print("X shape:", X.shape)
print("y shape:", y.shape)

# ======================
# 4. LSTM 모델 생성
# ======================
model = Sequential()
model.add(LSTM(64, input_shape=(seq_length, 45)))
model.add(Dense(45, activation='sigmoid'))

model.compile(
    loss='binary_crossentropy',
    optimizer='adam'
)

# ======================
# 5. 학습
# ======================
model.fit(X, y, epochs=20, batch_size=16)

# ======================
# 6. 예측
# ======================
last_seq = data[-seq_length:]
last_seq = last_seq.reshape(1, seq_length, 45)

pred = model.predict(last_seq)[0]

# ======================
# 7. 번호별 예측 확률 순위 출력
# ======================
sorted_indices = np.argsort(pred)[::-1]  # 내림차순

print("\n📊 번호별 예측 확률 순위:")

for rank, idx in enumerate(sorted_indices):
    number = idx + 1
    probability = pred[idx]
    print(f"{rank+1:2d}위: 번호 {number:2d} → 확률 {probability:.4f}")

# ======================
# 8. 추천 번호 생성
# ======================
# 상위 15개 후보
top_indices = np.argsort(pred)[-15:]

print("\n🎯 추천 번호 5세트:")

for i in range(5):
    recommended = np.random.choice(top_indices, 6, replace=False)
    recommended = sorted([int(i+1) for i in recommended])
    print(f"{i+1}번:", recommended)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
전처리 데이터:
   n1  n2  n3  n4  n5  n6
0   1  17  20  24  30  41
1   4  26  28  29  33  40
2  17  20  29  35  38  44
3   1  21  24  26  29  42
4   3  14  24  33  35  36
X shape: (1119, 10, 45)
y shape: (1119, 45)
Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


70/70 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.4786
Epoch 2/20
70/70 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.3936
Epoch 3/20
70/70 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.3932
Epoch 4/20
70/70 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.3930
Epoch 5/20
70/70 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.3925
Epoch 6/20
70/70 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.3922
Epoch 7/20
70/70 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.3920
Epoch 8/20
70/70 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.3916
Epoch 9/20
70/70 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.3909
Epoch 10/20
70/70 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.3905
Epoch 11/20
70/70 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.3901
Epoch 12/20
70/70 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.3897
Epoch 13/20
70/70 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.3889
Epoch 14/20
70/70 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.3885
Epoch 15/20
70/70 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.3875
Epoch 16/20
70/70 ━━━━━━━

# 자주 나온 번호 기반 추출


In [3]:
import pandas as pd
import numpy as np

# ======================
# 0. Google Drive 연결
# ======================
drive.mount('/content/drive')

# ======================
# 1. 원본 데이터 로드 & 전처리
# ======================
# Drive에 저장된 lotto.csv 읽기
raw_df = pd.read_csv("/content/drive/MyDrive/lotto.csv", header=None)

# 당첨번호만 추출
df = raw_df.iloc[:, 1:7]

# ======================
# 2. 최근 3년 (약 150회)
# ======================
recent_df = df.tail(150)

# ======================
# 3. 번호 빈도 계산
# ======================
counts = np.zeros(45)

for row in recent_df.values:
    for num in row:
        counts[int(num)-1] += 1

# ======================
# 4. 기본 출력
# ======================
print("📊 최근 3년 번호 출현 빈도\n")

for i in range(45):
    print(f"{i+1}번 : {int(counts[i])}회")

# ======================
# 5. 빈도순 정렬 출력
# ======================
print("\n🔥 많이 나온 번호 순\n")

sorted_idx = np.argsort(counts)[::-1]  # 내림차순

for i in sorted_idx:
    print(f"{i+1}번 : {int(counts[i])}회")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📊 최근 3년 번호 출현 빈도

1번 : 21회
2번 : 11회
3번 : 26회
4번 : 17회
5번 : 13회
6번 : 26회
7번 : 28회
8번 : 18회
9번 : 19회
10번 : 18회
11번 : 19회
12번 : 21회
13번 : 24회
14번 : 18회
15번 : 21회
16번 : 27회
17번 : 17회
18번 : 15회
19번 : 25회
20번 : 18회
21번 : 22회
22번 : 15회
23번 : 19회
24번 : 21회
25번 : 14회
26번 : 21회
27번 : 25회
28번 : 24회
29번 : 18회
30번 : 28회
31번 : 21회
32번 : 15회
33번 : 25회
34번 : 18회
35번 : 22회
36번 : 19회
37번 : 24회
38번 : 28회
39번 : 15회
40번 : 23회
41번 : 15회
42번 : 15회
43번 : 14회
44번 : 20회
45번 : 17회

🔥 많이 나온 번호 순

38번 : 28회
7번 : 28회
30번 : 28회
16번 : 27회
6번 : 26회
3번 : 26회
33번 : 25회
27번 : 25회
19번 : 25회
28번 : 24회
37번 : 24회
13번 : 24회
40번 : 23회
35번 : 22회
21번 : 22회
1번 : 21회
15번 : 21회
12번 : 21회
24번 : 21회
26번 : 21회
31번 : 21회
44번 : 20회
11번 : 19회
23번 : 19회
9번 : 19회
36번 : 19회
34번 : 18회
8번 : 18회
10번 : 18회
20번 : 18회
29번 : 18회
14번 : 18회
17번 : 17회
45번 : 17회
4번 : 17회
41번 : 15회
42번 : 15회
18번 : 15회
39번 : 15회
32번 : 15회
22번 